# KORE 데이터 작업 (B·C용, GPU 필요 없음)

파이썬을 내 컴퓨터에 설치하지 않고 Colab에서 데이터 명령을 돌린다. 순서: 1 → 2 → (B는 3-B, C는 3-C) → 4.
결과 파일은 내려받아서 GitHub 웹에서 올린다. Colab에서 직접 push하지 않는다(토큰이 읽기 전용이다).

## 1. 준비

In [ ]:
# 1. 준비: 최신 main 받기와 설치
import os, subprocess, sys, time

REPO_URL = 'https://github.com/OWNER/kore.git'  # 팀 저장소 주소로 바꾼다(Secrets에 KORE_REPO_URL을 저장했으면 그대로 둬도 된다)
try:
    from google.colab import userdata
    REPO_URL = userdata.get('KORE_REPO_URL') or REPO_URL
except Exception:
    pass
BRANCH = 'main'
WORKDIR = '/content/kore'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')  # Colab 왼쪽 Secrets에 저장한 읽기 전용 토큰(비공개 저장소일 때만)
    except Exception:
        return None

def sh(cmd, cwd=None, stream=False, secret=None):
    """명령을 실행하고 출력을 보여 준다. secret이 출력에 섞이면 가린다."""
    mask = (lambda s: s.replace(secret, '***')) if secret else (lambda s: s)
    if stream:
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            print(mask(line), end='')
        p.wait()
        rc = p.returncode
    else:
        r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        print(mask(r.stdout[-3000:] + r.stderr[-1500:]))
        rc = r.returncode
    if rc != 0:
        raise RuntimeError('명령 실패(코드 %s): %s' % (rc, mask(cmd)))

if 'OWNER' in REPO_URL:
    raise SystemExit('REPO_URL을 팀 저장소 주소로 바꾼 뒤 다시 실행한다.')
token = _token()
url = REPO_URL.replace('https://', 'https://x-access-token:%s@' % token) if token else REPO_URL
if os.path.isdir(WORKDIR):
    sh('git fetch origin && git checkout ' + BRANCH + ' && git reset --hard origin/' + BRANCH, cwd=WORKDIR, secret=token)
else:
    sh('git clone --branch ' + BRANCH + ' ' + url + ' ' + WORKDIR, secret=token)
sh(sys.executable + ' -m pip install -q -e .', cwd=WORKDIR)
os.makedirs(WORKDIR + '/raw', exist_ok=True)
print('준비 끝. 최신 main을 받았다. GPU는 필요 없다(런타임 유형: CPU).')

## 2. 파일 올리기

In [ ]:
# 2. 내 컴퓨터의 파일을 Colab으로 올린다
# B: 원문 txt → raw/ 폴더,  C: evaluation_raw.csv → data/ 폴더
import os, shutil
from google.colab import files
WORKDIR = '/content/kore'
up = files.upload()
for name in up:
    dest = 'data' if name.endswith('.csv') else 'raw'
    shutil.move(name, os.path.join(WORKDIR, dest, name))
    print('올림:', dest + '/' + name)

## 3-B. 문서 올리기 (B)
`DRY_RUN = True`로 먼저 보고, 잘린 모양이 맞으면 `False`로 바꿔 다시 실행한다.

In [ ]:
# 3-B. 원문을 문서 CSV로
import os, subprocess, sys, time

REPO_URL = 'https://github.com/OWNER/kore.git'  # 팀 저장소 주소로 바꾼다(Secrets에 KORE_REPO_URL을 저장했으면 그대로 둬도 된다)
try:
    from google.colab import userdata
    REPO_URL = userdata.get('KORE_REPO_URL') or REPO_URL
except Exception:
    pass
BRANCH = 'main'
WORKDIR = '/content/kore'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')  # Colab 왼쪽 Secrets에 저장한 읽기 전용 토큰(비공개 저장소일 때만)
    except Exception:
        return None

def sh(cmd, cwd=None, stream=False, secret=None):
    """명령을 실행하고 출력을 보여 준다. secret이 출력에 섞이면 가린다."""
    mask = (lambda s: s.replace(secret, '***')) if secret else (lambda s: s)
    if stream:
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            print(mask(line), end='')
        p.wait()
        rc = p.returncode
    else:
        r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        print(mask(r.stdout[-3000:] + r.stderr[-1500:]))
        rc = r.returncode
    if rc != 0:
        raise RuntimeError('명령 실패(코드 %s): %s' % (rc, mask(cmd)))

os.chdir(WORKDIR)
# ---- 여기만 바꾼다 ----
INPUT = 'raw/law_labor.txt'               # 2번 셀로 올린 원문
DOMAIN = 'law'                            # law / guide / wiki
ID_PREFIX = 'law_labor'                   # 문서 id 앞부분(파일마다 다르게)
SOURCE_URL = 'https://www.law.go.kr/...'  # 원문 주소
LICENSE = '저작권법 제7조(법령)'            # 직접 확인한 이용 조건. 모르면 unknown
MODE = 'article'                          # 법령은 article, 제목이 있는 안내문은 heading, 그 밖은 paragraph
ACCESSED = '2026-10-24'                   # 원문을 연 날짜
DRY_RUN = True                            # 먼저 True로 잘리는 모양을 보고, 괜찮으면 False로 바꿔 다시 실행
# ----------------------
cmd = (sys.executable + ' -m kore ingest --input "%s" --domain %s --id-prefix %s --source-url "%s" --license "%s" '
       '--mode %s --out data/knowledge.csv --append --accessed-date %s') % (INPUT, DOMAIN, ID_PREFIX, SOURCE_URL, LICENSE, MODE, ACCESSED)
sh(cmd + (' --dry-run' if DRY_RUN else ''), stream=True)
if not DRY_RUN:
    sh(sys.executable + ' -m kore validate --knowledge data/knowledge.csv', stream=True)
    sh(sys.executable + ' -m kore stats --knowledge data/knowledge.csv --stage week1', stream=True)
    print('다음: 4번 셀로 knowledge.csv와 source_log.csv를 내려받아 GitHub에 올린다(START.md 6절).')

## 3-C. 질문 나누기와 점검 (C)
`evaluation_raw.csv`를 2번 셀로 올린 뒤 실행한다.

In [ ]:
# 3-C. 분할과 점검
import os, subprocess, sys, time

REPO_URL = 'https://github.com/OWNER/kore.git'  # 팀 저장소 주소로 바꾼다(Secrets에 KORE_REPO_URL을 저장했으면 그대로 둬도 된다)
try:
    from google.colab import userdata
    REPO_URL = userdata.get('KORE_REPO_URL') or REPO_URL
except Exception:
    pass
BRANCH = 'main'
WORKDIR = '/content/kore'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')  # Colab 왼쪽 Secrets에 저장한 읽기 전용 토큰(비공개 저장소일 때만)
    except Exception:
        return None

def sh(cmd, cwd=None, stream=False, secret=None):
    """명령을 실행하고 출력을 보여 준다. secret이 출력에 섞이면 가린다."""
    mask = (lambda s: s.replace(secret, '***')) if secret else (lambda s: s)
    if stream:
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            print(mask(line), end='')
        p.wait()
        rc = p.returncode
    else:
        r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        print(mask(r.stdout[-3000:] + r.stderr[-1500:]))
        rc = r.returncode
    if rc != 0:
        raise RuntimeError('명령 실패(코드 %s): %s' % (rc, mask(cmd)))

os.chdir(WORKDIR)
FIRST_TIME = not os.path.exists('data/evaluation.csv')   # 처음이면 새로 나누고, 이미 있으면 기존 분할을 지킨다
cmd = sys.executable + ' -m kore split --questions data/evaluation_raw.csv --knowledge data/knowledge.csv --out data/evaluation.csv --seed 0'
if not FIRST_TIME:
    cmd += ' --keep-from data/evaluation.csv'
sh(cmd, stream=True)
sh(sys.executable + ' -m kore validate --knowledge data/knowledge.csv --questions data/evaluation.csv', stream=True)
sh(sys.executable + ' -m kore stats --knowledge data/knowledge.csv --questions data/evaluation.csv --stage week1', stream=True)
print('[실패] (필수)가 남아 있으면 고쳐서 다시 올린다. 통과하면 4번 셀로 evaluation.csv를 내려받는다.')

## 4. 내려받기

In [ ]:
# 4. 결과 파일 내려받기 → GitHub 웹에서 data/ 폴더에 올린다(START.md 6절)
from google.colab import files
WHO = 'B'   # B면 knowledge.csv와 source_log.csv, C면 evaluation.csv와 evaluation_raw.csv
names = ['knowledge.csv', 'source_log.csv'] if WHO == 'B' else ['evaluation.csv', 'evaluation_raw.csv']
for n in names:
    files.download('/content/kore/data/' + n)